# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore, process, and analyze a rich dataset described by a Croissant schema. All references to dataset entities use the corresponding `@id` fields as per the FAIR principles and Croissant model.

### Dataset Source
This dataset is described via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will retrieve the Croissant schema and allow you to review available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Metadata object

print(f"Dataset Title: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Let's review available `RecordSet` entries and their `@id`s. Each record set contains fields (columns). We'll enumerate all record sets and the ids of their fields for further reference.

In [ ]:
# List all RecordSets and their field @ids
print("RecordSets in dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields (@id):")
    for field in rs.fields:
        print(f"    - {field.name}: {field.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We'll use the record set and field `@id`s obtained above.

In [ ]:
# For illustration, select the first RecordSet's @id
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id  # Always reference by @id

# Extract all records for all record sets into DataFrames
dataframes = {}
for rs in record_sets:
    # using the @id string
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df

print(f"Columns for RecordSet '{main_record_set.name}' (id: {main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field using its `@id` (see section 2), filter for high abundance protein entries, normalize values, and optionally group by a categorical field.

**Reminder:** Replace `numeric_field_id` and `group_field_id` below with the exact `@id` values of interest from the above lists.

In [ ]:
# Example: Use a known numeric field @id. (Replace the field id as needed)
# Let's programmatically pick a numeric field for demonstration.
numeric_field_id = None
for field in main_record_set.fields:
    if getattr(field, 'data_type', None) in ('Number', 'Float', 'Integer'):
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    raise ValueError("No numeric field found; please inspect the field list and choose a numeric field @id.")

print(f"Selected numeric field for analysis: {numeric_field_id}")

main_df = dataframes[main_record_set_id]
# Remove missing values for numeric analysis
filtered_df = main_df.copy()
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').notnull()]
filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id])

# Filter for records with high numeric value
threshold = filtered_df[numeric_field_id].quantile(0.75)  # 75th percentile
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile): {len(filtered_df)} rows.")

# Normalize the chosen numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std(ddof=0)

print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group by a categorical field if present
group_field_id = None
for field in main_record_set.fields:
    # Skip numeric fields
    if getattr(field, 'data_type', None) not in ('Number', 'Float', 'Integer'):
        group_field_id = field.id
        break

if group_field_id and group_field_id in main_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"{numeric_field_id}_mean")
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distribution of the numeric field and its normalized form, and relationships with a group field where applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Normalized histogram
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id} (filtered)")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel('Count')
plt.show()

# Boxplot by group if group_field_id is available
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and inspected the Mass Spectrometry dataset using the `mlcroissant` library, referencing all elements by their `@id` for reproducibility and precision. We listed available record sets and fields, loaded data for a chosen record set, performed basic exploration and filtering using numeric fields, normalized values, and visualized distributions and group effects.

**Next steps:**
- Explore additional record sets or fields for more in-depth analyses.
- Apply domain-specific filtering or machine learning workflows on the extracted DataFrames.
- Use the @id-based extraction model for robust, reproducible data analysis pipelines.
